<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/main/Prac_12_Regression_Machine_Learning_Case_Study_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 12: Regression Machine Learning Case Study Project

How do you work through a predictive modeling machine learning problem end-to-end? In this
lesson you will work through a case study regression predictive modeling problem in Python
including each step of the applied machine learning process. After completing this project, you
will know:
- How to work through a regression predictive modeling problem end-to-end.
- How to use data transforms to improve model performance.
- How to use algorithm tuning to improve model performance.
- How to use ensemble methods and tuning of ensemble methods to improve model performance.

## Problem Definition

For this project, we will investigate a synthetic Naval Maintenance dataset. Each record in the dataset represents a cargo ship and its technical and operational characteristics. The data simulates realistic information commonly found in the maritime engineering field, focusing on factors that influence a vessel's annual maintenance cost. The attributes are defined as follows:

- **FUEL_CONSUMP**: Average daily fuel consumption (in tons per day).
- **CARGO_CAPACITY**: Maximum cargo capacity (in thousand metric tons).
- **ENGINE_POWER**: Main engine power output (in megawatts).
- **HAS_DUAL_PROP**: Binary indicator of whether the vessel has a dual propeller system (0 = no, 1 = yes).
- **NOX_EMISSIONS**: Emissions of nitrogen oxides from the engine (in grams per kilowatt-hour).
- **CREW_SIZE**: Average number of crew members on board.
- **AGE_YEARS**: Age of the vessel (in years since commissioning).
- **PORT_DISTANCE**: Average distance to the vessel’s home port during operation (in nautical miles).
- **NAV_EQUIP_LEVEL**: Level of onboard navigation and electronic systems (scale from 1 to 10).
- **REG_FEES**: Annual registration and compliance fees (in thousands of USD).
- **MAINT_CREW_RATIO**: Ratio of maintenance personnel to total crew.
- **PAINT_COVERAGE**: Index representing the quality and coverage of anti-corrosive coating (scale 0–100).
- **RUST_AREA_PERCENT**: Percentage of the ship’s surface area affected by active corrosion or rust.
- **MAINT_COST**: **Target variable** — Estimated annual maintenance cost for the vessel (in thousands of USD).




In [ ]:
# Load libraries
import numpy
from numpy import arange
from matplotlib import pyplot
from pandas import read_csv
from pandas import set_option
from pandas.plotting import scatter_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.model_selection import KFold
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.linear_model import ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import AdaBoostRegressor
from sklearn.metrics import mean_squared_error

In [1]:
import pandas as pd
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)

# Number of samples (same as Boston dataset)
n_samples = 506

# Generate synthetic data (based on plausible naval engineering ranges)
data = {
    "FUEL_CONSUMP": np.round(np.random.normal(loc=30, scale=10, size=n_samples), 2),  # tons/day
    "CARGO_CAPACITY": np.round(np.random.normal(loc=50, scale=20, size=n_samples), 1),  # thousand tons
    "ENGINE_POWER": np.round(np.random.normal(loc=25, scale=5, size=n_samples), 2),  # MW
    "HAS_DUAL_PROP": np.random.choice([0, 1], size=n_samples),  # 0 or 1
    "NOX_EMISSIONS": np.round(np.random.normal(loc=5.0, scale=1.5, size=n_samples), 2),  # g/kWh
    "CREW_SIZE": np.round(np.random.normal(loc=25, scale=5, size=n_samples), 0),  # number of crew
    "AGE_YEARS": np.round(np.random.normal(loc=15, scale=10, size=n_samples), 1),  # years
    "PORT_DISTANCE": np.round(np.random.normal(loc=150, scale=50, size=n_samples), 1),  # NM
    "NAV_EQUIP_LEVEL": np.random.randint(1, 11, size=n_samples),  # scale 1 to 10
    "REG_FEES": np.round(np.random.normal(loc=15, scale=5, size=n_samples), 2),  # thousands USD
    "MAINT_CREW_RATIO": np.round(np.random.normal(loc=0.2, scale=0.05, size=n_samples), 2),  # ratio
    "PAINT_COVERAGE": np.round(np.random.normal(loc=85, scale=10, size=n_samples), 1),  # index 0–100
    "RUST_AREA_PERCENT": np.round(np.random.normal(loc=12, scale=7, size=n_samples), 2),  # %
}

# Generate the target variable (MAINT_COST) based on a weighted combination + noise
# This simulates a plausible regression target
weights = {
    "FUEL_CONSUMP": 1.5,
    "AGE_YEARS": 2.0,
    "ENGINE_POWER": 1.2,
    "RUST_AREA_PERCENT": 3.0,
    "CREW_SIZE": 0.5,
    "HAS_DUAL_PROP": 5.0,
    "REG_FEES": 1.0
}
noise = np.random.normal(0, 10, n_samples)

# Calculate maintenance cost (in thousands USD)
maint_cost = (
    weights["FUEL_CONSUMP"] * data["FUEL_CONSUMP"] +
    weights["AGE_YEARS"] * data["AGE_YEARS"] +
    weights["ENGINE_POWER"] * data["ENGINE_POWER"] +
    weights["RUST_AREA_PERCENT"] * data["RUST_AREA_PERCENT"] +
    weights["CREW_SIZE"] * data["CREW_SIZE"] +
    weights["HAS_DUAL_PROP"] * data["HAS_DUAL_PROP"] +
    weights["REG_FEES"] * data["REG_FEES"] +
    noise
)

# Add target to data
data["MAINT_COST"] = np.round(maint_cost, 2)

# Create DataFrame
df = pd.DataFrame(data)

# Save to CSV
df.to_csv("naval_maintenance_dataset.csv", index=False)

print("Dataset saved as 'naval_maintenance_dataset.csv'")


Dataset saved as 'naval_maintenance_dataset.csv'
